In [1]:

!pip install pymunk -q

import os
import cv2
import pymunk
import random
import zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm

dataset_dir = "physics_dataset"
images_dir = os.path.join(dataset_dir, "images")
os.makedirs(images_dir, exist_ok=True)

def generate_collision_scenario(sim_id):
    WIDTH, HEIGHT = 400, 400

    space = pymunk.Space()
    space.gravity = (0, 0)

    static_lines = [
        pymunk.Segment(space.static_body, (0, 0), (WIDTH, 0), 2),
        pymunk.Segment(space.static_body, (WIDTH, 0), (WIDTH, HEIGHT), 2),
        pymunk.Segment(space.static_body, (WIDTH, HEIGHT), (0, HEIGHT), 2),
        pymunk.Segment(space.static_body, (0, HEIGHT), (0, 0), 2)
    ]
    for line in static_lines:
        line.elasticity = 1.0
        space.add(line)

    radius, mass = 15, 1
    moment = pymunk.moment_for_circle(mass, 0, radius)

    def spawn_safe_body():
        x = random.randint(radius + 10, WIDTH - radius - 10)
        y = random.randint(radius + 10, HEIGHT - radius - 10)
        vx = random.uniform(-200, 200)
        vy = random.uniform(-200, 200)

        body = pymunk.Body(mass, moment)
        body.position = (x, y)
        body.velocity = (vx, vy)
        shape = pymunk.Circle(body, radius)
        shape.elasticity = 1.0
        return body, shape

    body_A, shape_A = spawn_safe_body()
    body_B, shape_B = spawn_safe_body()

    while np.linalg.norm(np.array(body_A.position) - np.array(body_B.position)) < radius * 2 + 10:
        body_B, shape_B = spawn_safe_body()

    space.add(body_A, shape_A)
    space.add(body_B, shape_B)

    img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

    pos_A = (int(body_A.position.x), int(body_A.position.y))
    pos_B = (int(body_B.position.x), int(body_B.position.y))

    cv2.circle(img, pos_A, radius, (255, 0, 0), -1)
    cv2.circle(img, pos_B, radius, (0, 0, 255), -1)
    arrow_scale = 0.3
    target_A = (int(pos_A[0] + body_A.velocity.x * arrow_scale), int(pos_A[1] + body_A.velocity.y * arrow_scale))
    target_B = (int(pos_B[0] + body_B.velocity.x * arrow_scale), int(pos_B[1] + body_B.velocity.y * arrow_scale))

    cv2.arrowedLine(img, pos_A, target_A, (255, 255, 255), 2, tipLength=0.3)
    cv2.arrowedLine(img, pos_B, target_B, (255, 255, 255), 2, tipLength=0.3)

    img_filename = f"scene_{sim_id}.png"
    img_path = os.path.join(images_dir, img_filename)
    cv2.imwrite(img_path, img)

    will_collide = 0
    duration_seconds = 3.0
    fps = 60

    for _ in range(int(duration_seconds * fps)):
        space.step(1.0 / fps)
        distance = np.linalg.norm(np.array(body_A.position) - np.array(body_B.position))
        if distance <= (radius * 2 + 2):
            will_collide = 1
            break

    return {
        "image_file": img_filename,
        "will_collide": will_collide
    }

NUM_SAMPLES = 2000
print(f"Generating {NUM_SAMPLES} physics scenarios...")

dataset_records = []
for i in tqdm(range(NUM_SAMPLES)):
    record = generate_collision_scenario(i)
    dataset_records.append(record)
df = pd.DataFrame(dataset_records)
csv_path = os.path.join(dataset_dir, "labels.csv")
df.to_csv(csv_path, index=False)
print(f"\nDataset saved! Class balance:\n{df['will_collide'].value_counts(normalize=True)}")

print("\nCompressing dataset into a single zip file...")
zip_filename = "physics_dataset.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(dataset_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(dataset_dir))
            zipf.write(file_path, arcname)

print(f"Done! You can now download {zip_filename} from the Colab file explorer.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.9 MB/s eta 0:00:00
Generating 2000 physics scenarios...


100%|██████████| 2000/2000 [00:09<00:00, 220.78it/s]



Dataset saved! Class balance:
will_collide
0    0.7565
1    0.2435
Name: proportion, dtype: float64

Compressing dataset into a single zip file...
Done! You can now download physics_dataset.zip from the Colab file explorer.


In [6]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

shutil.copy("physics_dataset.zip", "/content/drive/MyDrive/physics_dataset.zip")
print("Saved to your Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to your Google Drive!


In [10]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

!unzip -o -q "/content/drive/MyDrive/physics_dataset.zip" -d "/content/"

class PhysicsDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data_labels = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_labels)

    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data_labels.iloc[idx]['image_file'])
        image = Image.open(img_name).convert('RGB')

        label = float(self.data_labels.iloc[idx]['will_collide'])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = PhysicsDataset(
    csv_file='/content/physics_dataset/labels.csv',
    img_dir='/content/physics_dataset/images',
    transform=transform
)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"\nDataset successfully loaded!")
print(f"Training on {len(train_dataset)} scenarios | Testing on {len(val_dataset)} scenarios")


Dataset successfully loaded!
Training on 1600 scenarios | Testing on 400 scenarios


In [11]:
import torch.nn as nn
from torchvision import models

class PhysicsPredictor(nn.Module):
    def __init__(self):
        super(PhysicsPredictor, self).__init__()

        self.encoder = models.resnet18(weights='DEFAULT')

        num_latent_features = self.encoder.fc.in_features

        self.encoder.fc = nn.Identity()

        self.readout_head = nn.Sequential(
            nn.Linear(num_latent_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        latent_space = self.encoder(x)
        prediction = self.readout_head(latent_space)
        return prediction.squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PhysicsPredictor().to(device)

print(f"Physics Model built and loaded onto: {device}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 204MB/s]


Physics Model built and loaded onto: cuda


In [12]:
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 5

print("Starting Training Sequence...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Error (Loss): {avg_loss:.4f}")

print("Training Complete! Model has learned the physics rules.")

Starting Training Sequence...
Epoch [1/5] - Error (Loss): 0.5604
Epoch [2/5] - Error (Loss): 0.5368
Epoch [3/5] - Error (Loss): 0.5183
Epoch [4/5] - Error (Loss): 0.5049
Epoch [5/5] - Error (Loss): 0.4861
Training Complete! Model has learned the physics rules.
